In [1]:

# Imports needed 
import os
import pandas as pd
import requests
from Bio import SeqIO

def reverse_complement(dna_seq):
    """ 
    Return the reverse complement of a DNA sequence. 
    Arguments:
        dna_seq - DNA input sequence as a string
        
    Returns:
        reverse_complement - Reverse complement of DNA input sequence 
    
    """
    complement = {'a': 't', 'A':'T', 't': 'a', 'T':'A','c': 'g','C':'G', 'g': 'c', 'G':'C'}
    
    reverse_complement_seq = ''.join(complement.get(base, base) for base in reversed(dna_seq))
        
    return reverse_complement_seq

def translate(dna_seq):
    """
    Generates a protein sequence from a DNA sequence inputted as a string. It will only convert if the exon is in upper
    case

    Arguments:
        dna_seq(str) - input DNA sequence
        
    Returns:
        protein_sequence(str) - output protein sequence 
    """
    
    #Generate dictionary of codons for dna to protein translation
    
    codon_table = {
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',  # Alanine
    'TGT': 'C', 'TGC': 'C',                          # Cysteine
    'GAT': 'D', 'GAC': 'D',                          # Aspartic acid
    'GAA': 'E', 'GAG': 'E',                          # Glutamic acid
    'TTT': 'F', 'TTC': 'F',                          # Phenylalanine
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',  # Glycine
    'CAT': 'H', 'CAC': 'H',                          # Histidine
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I',              # Isoleucine
    'AAA': 'K', 'AAG': 'K',                          # Lysine
    'TTA': 'L', 'TTG': 'L', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',  # Leucine
    'ATG': 'M',                                      # Methionine (Start codon)
    'AAT': 'N', 'AAC': 'N',                          # Asparagine
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',  # Proline
    'CAA': 'Q', 'CAG': 'Q',                          # Glutamine
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R', 'AGA': 'R', 'AGG': 'R',  # Arginine
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 'AGT': 'S', 'AGC': 'S',  # Serine
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',  # Threonine
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',  # Valine
    'TGG': 'W',                                      # Tryptophan
    'TAT': 'Y', 'TAC': 'Y',                          # Tyrosine
    'TAA': '*', 'TAG': '*', 'TGA': '*'               # Stop codons
    }
    
    # Initialize the protein sequence
    protein = ""

    # Iterate over the DNA sequence in steps of 3 (codon length)
    for i in range(0, len(dna_seq) - len(dna_seq) % 3, 3):
        
        # Define a codon as 3 bases
        codon = dna_seq[i:i + 3]
        # Translate the codon into an amino acid based on the dictionary
        if codon.isupper():
            protein += codon_table.get(codon, '?')  # '?' is a placeholder for unknown codons
        else:
            protein += 'x'

    return protein

def extract_genomic_information_from_uniprot_id(uniprot_id):
    '''
    Takes in uniprot ID as a string input and pings the Uniprot API to extract genomic coordinates of the protein and exons.
    Metadata such as name, taxID, protein sequence, genome assembly name,  ENSEMBL GeneID, ENSEMBL Transcript ID and ENSEMBL Translations IDs is included alongside the extracted coordinates.  

    Args:
    uniprot_id (str): uniprot ID

    Returns:
    genomic_information (pd.DataFrame): DataFrame containing genomic coordinates of the protein of interest alongside exon positions and metadata 
    '''
    genomic_information = pd.DataFrame()
    try:
        print(f'Searching for UniProt ID: {uniprot_id}')
        requestURL_protein = f"https://www.ebi.ac.uk/proteins/api/coordinates/{uniprot_id}"
        response_protein = requests.get(requestURL_protein, headers={"Accept": "application/json"})
        
        # Check if the request was successful
        response_protein.raise_for_status()
        
        # Load JSON response
        response_protein = response_protein.json()
        
        # Check if response is not empty
        if response_protein:
            response_protein_normalise = pd.json_normalize(
                response_protein, 
                record_path=['gnCoordinate', 'genomicLocation', 'exon'], 
                meta=['accession', 'name', 'taxid', 'sequence', 
                      ['gnCoordinate', 'genomicLocation', 'chromosome'], 
                      ['gnCoordinate', 'genomicLocation', 'start'], 
                      ['gnCoordinate', 'genomicLocation', 'end'], 
                      ['gnCoordinate', 'genomicLocation', 'reverseStrand'], 
                      ['gnCoordinate', 'genomicLocation', 'nucleotideId'], 
                      ['gnCoordinate', 'genomicLocation', 'assemblyName'], 
                      ['gnCoordinate', 'ensemblGeneId'], 
                      ['gnCoordinate', 'ensemblTranscriptId'], 
                      ['gnCoordinate', 'ensemblTranslationId']],
                record_prefix='exon_'
            )

            # Group and aggregate exon information
            response_protein_normalise = response_protein_normalise.groupby([
                'accession', 'name', 'taxid', 'sequence', 
                'gnCoordinate.genomicLocation.chromosome', 
                'gnCoordinate.genomicLocation.start', 
                'gnCoordinate.genomicLocation.end', 
                'gnCoordinate.genomicLocation.reverseStrand', 
                'gnCoordinate.genomicLocation.nucleotideId', 
                'gnCoordinate.genomicLocation.assemblyName', 
                'gnCoordinate.ensemblGeneId', 
                'gnCoordinate.ensemblTranscriptId', 
                'gnCoordinate.ensemblTranslationId'
            ]).agg({
                'exon_id': lambda x: ','.join(map(str, x)),
                'exon_proteinLocation.begin.position': lambda x: ','.join(map(str, x)),                    
                'exon_proteinLocation.end.position': lambda x: ','.join(map(str, x)),
                'exon_genomeLocation.begin.position': lambda x: ','.join(map(str, x)),                    
                'exon_genomeLocation.end.position': lambda x: ','.join(map(str, x))
            }).reset_index()

            # Concatenate to the main DataFrame
            genomic_information = pd.concat([genomic_information, response_protein_normalise], ignore_index=True)
        else:
            print(f"No data found for UniProt ID: {uniprot_id}")
            
    except Exception as e:
        print(f"An error occurred: {e}")

    return genomic_information
    
# Identify features
def find_feature(record, search_term):
    term = search_term.lower().strip()
    for feature in record.features:
        # Check the type first
        if feature.type.lower() == term:
            return feature
        # Then check all qualifiers
        if any(term in str(v).lower() for vals in feature.qualifiers.values() for v in vals):
            return feature
    return None

# Extract seqs
def extract_slice(seq, coord1, coord2, strand):
    start = int(min(coord1, coord2))
    end = int(max(coord1, coord2))
    extracted_seq = seq[start:end]

    if strand == 'Reverse':
        return str(extracted_seq.reverse_complement()).lower()
    return str(extracted_seq).lower()

# Calculate distances between features and return the minimum
def get_min_distance(feat1, feat2):
    return min(
        abs(feat1.location.start - feat2.location.start),
        abs(feat1.location.end - feat2.location.end),
        abs(feat1.location.start - feat2.location.end),
        abs(feat1.location.end - feat2.location.start)
    )

# Process GenBank files
def process_genbank(name, GENBANK_DIR, read_length, protein_strand):
    gb_path = os.path.join(GENBANK_DIR, f"{name}.gb")
    if not os.path.exists(gb_path):
        gb_path = os.path.join(GENBANK_DIR, f"{name}.gbk")
        if not os.path.exists(gb_path):
            return None, "GenBank file not found"

    record = SeqIO.read(gb_path, "genbank")
    
    # Step 1: Locate all four required features
    target_feat = find_feature(record, "target_codon")
    recod_feat = find_feature(record, "recodonised")
    fwd_feat = find_feature(record, "fwd")
    rev_feat = find_feature(record, "rev")

    missing = []
    if not target_feat: missing.append("target_codon")
    if not recod_feat: missing.append("recodonised")
    if not fwd_feat: missing.append("fwd")
    if not rev_feat: missing.append("rev")
    
    if missing:
        return None, f"Missing annotations: {', '.join(missing)}"

    # Step 2: calculate distances between primers and target codon to identify which strand the target codon is on. 
    dist_fwd = get_min_distance(target_feat, fwd_feat)
    print('Distance between target codon and forward primer:', dist_fwd)
    dist_rev = get_min_distance(target_feat, rev_feat)
    print('Distance between target codon and reverse primer:', dist_rev)
    if dist_fwd < dist_rev:
        isp_primer_strand = 'Forward'
        print('ISP primer strand:', isp_primer_strand)
    else:
        isp_primer_strand = 'Reverse'
        print('ISP primer strand:', isp_primer_strand)

    # Step 3: Extract sequences based on the leading seq strand (i.e. fwd or rev)
    if isp_primer_strand == 'Forward':
        # FORWARD STRAND
        # Innoculum features (capped at 150 total sublibrary primer(20) + homology arm (60) + recodonised region to target codon (X) + target codon (3) + recodonised region after target codon (150 - 20 - 60 - X - 3))
        # Note: here we don't add the sublibrary primer 
        inn_prefix = extract_slice(seq = record.seq, coord1 = recod_feat.location.start - 60, coord2 = target_feat.location.start, strand = isp_primer_strand)
        print('Innoculum prefix:', inn_prefix)
        print('Length of innoculum prefix:', len(inn_prefix))
        codon_seq = extract_slice(seq = record.seq, coord1 = target_feat.location.start, coord2 = target_feat.location.end, strand = isp_primer_strand)
        print('Target codon sequence:', codon_seq)
        print('Length of target codon sequence:', len(codon_seq))
        # Calculate space left for suffix
        allowed_inn_suffix_len = read_length - 20 - len(inn_prefix) - len(codon_seq)
        inn_suffix_end = target_feat.location.end + allowed_inn_suffix_len
        inn_suffix = extract_slice(seq = record.seq, coord1 = target_feat.location.end, coord2 = inn_suffix_end, strand = isp_primer_strand)
        print('Innoculum suffix:', inn_suffix)
        print('Length of innoculum suffix:', len(inn_suffix))

        # Note: there are rare cases where the space between guides is too great and the variant codon isn't captured in the 150 bp amplicon. The if statement below captures this case.
        if len(inn_prefix) + len(codon_seq) + len(inn_suffix) > 130:
            inn_prefix = extract_slice(seq = record.seq, coord1 = target_feat.location.end, coord2 = recod_feat.location.end + 60, strand = isp_primer_strand)
            print('Innoculum prefix:', inn_prefix)
            print('Length of innoculum prefix:', len(inn_prefix))
            codon_seq = extract_slice(seq = record.seq, coord1 = target_feat.location.start, coord2 = target_feat.location.end, strand = isp_primer_strand)
            print('Target codon sequence:', codon_seq)
            print('Length of target codon sequence:', len(codon_seq))
            # Calculate space left for suffix
            allowed_inn_suffix_len = read_length - 20 - len(inn_prefix) - len(codon_seq)
            print('Allowed innoculum suffix length:', allowed_inn_suffix_len)
            inn_suffix_start = target_feat.location.start - allowed_inn_suffix_len
            inn_suffix = extract_slice(seq = record.seq, coord1 = inn_suffix_start, coord2 = target_feat.location.start, strand = isp_primer_strand)
            print('Innoculum suffix:', inn_suffix)
            print('Length of innoculum suffix:', len(inn_suffix))

        # ISP features (capped at 150bp total - prefix + codon + suffix)
        isp_prefix = extract_slice(seq = record.seq, coord1 = fwd_feat.location.start, coord2 = target_feat.location.start, strand = isp_primer_strand)
        print('ISP prefix:', isp_prefix)
        print('Length of ISP prefix:', len(isp_prefix))
        # Calculate space left for suffix
        allowed_isp_suffix_len = read_length - len(isp_prefix) - len(codon_seq)
        isp_suffix_end = target_feat.location.end + allowed_isp_suffix_len
        isp_suffix = extract_slice(seq = record.seq, coord1 = target_feat.location.end, coord2 = isp_suffix_end, strand = isp_primer_strand)
        print('ISP suffix:', isp_suffix)
        print('Length of ISP suffix:', len(isp_suffix))
    
    else:
        # REVERSE STRAND
        # Innoculum features (capped at 150 total sublibrary primer(20) + homology arm (60) + recodonised region to target codon (X) + target codon (3) + recodonised region after target codon (150 - 20 - 60 - X - 3))
        # Note: here we don't add the sublibrary primer 
        inn_prefix = extract_slice(seq = record.seq, coord1 = target_feat.location.end, coord2 = recod_feat.location.end + 60, strand = isp_primer_strand)
        print('Innoculum prefix:', inn_prefix)
        print('Length of innoculum prefix:', len(inn_prefix))
        codon_seq = extract_slice(seq = record.seq, coord1 = target_feat.location.start, coord2 = target_feat.location.end, strand = isp_primer_strand)
        print('Target codon sequence:', codon_seq)
        print('Length of target codon sequence:', len(codon_seq))
        # Calculate space left for suffix
        allowed_inn_suffix_len = read_length - 20 - len(inn_prefix) - len(codon_seq)
        print('Allowed innoculum suffix length:', allowed_inn_suffix_len)
        inn_suffix_start = target_feat.location.start - allowed_inn_suffix_len
        inn_suffix = extract_slice(seq = record.seq, coord1 = inn_suffix_start, coord2 = target_feat.location.start, strand = isp_primer_strand)
        print('Innoculum suffix:', inn_suffix)
        print('Length of innoculum suffix:', len(inn_suffix))

        if len(inn_prefix) + len(codon_seq) + len(inn_suffix) > 130:
            # Innoculum features (capped at 150 total sublibrary primer(20) + homology arm (60) + recodonised region to target codon (X) + target codon (3) + recodonised region after target codon (150 - 20 - 60 - X - 3))
            # Note: here we don't add the sublibrary primer 
            inn_prefix = extract_slice(seq = record.seq, coord1 = recod_feat.location.start - 60, coord2 = target_feat.location.start, strand = isp_primer_strand)
            print('Innoculum prefix:', inn_prefix)
            print('Length of innoculum prefix:', len(inn_prefix))
            codon_seq = extract_slice(seq = record.seq, coord1 = target_feat.location.start, coord2 = target_feat.location.end, strand = isp_primer_strand)
            print('Target codon sequence:', codon_seq)
            print('Length of target codon sequence:', len(codon_seq))
            # Calculate space left for suffix
            allowed_inn_suffix_len = read_length - 20 - len(inn_prefix) - len(codon_seq)
            inn_suffix_end = target_feat.location.end + allowed_inn_suffix_len
            inn_suffix = extract_slice(seq = record.seq, coord1 = target_feat.location.end, coord2 = inn_suffix_end, strand = isp_primer_strand)
            print('Innoculum suffix:', inn_suffix)
            print('Length of innoculum suffix:', len(inn_suffix))

        # ISP features (capped at 150bp total)
        # Prefix starts at the physical 5' end of the rev primer (which is its higher coordinate: .end)
        isp_prefix = extract_slice(seq = record.seq, coord1 = target_feat.location.end, coord2 = rev_feat.location.end, strand = isp_primer_strand)
        print('ISP prefix:', isp_prefix)
        print('Length of ISP prefix:', len(isp_prefix))
        # Calculate space left for suffix
        allowed_isp_suffix_len = read_length - len(isp_prefix) - len(codon_seq)
        # Suffix towards the fwd primer
        isp_suffix_start = target_feat.location.start - allowed_isp_suffix_len
        isp_suffix = extract_slice(seq = record.seq, coord1 = isp_suffix_start, coord2 = target_feat.location.start, strand = isp_primer_strand)
        print('ISP suffix:', isp_suffix)
        print('Length of ISP suffix:', len(isp_suffix))
        
       
    # translate codon sequence to amino acid
    if isp_primer_strand == 'Forward':
        print('ISP forward primer is closest to the target codon so target codon will be translated directly.')    
        wt_codon = codon_seq.upper()
        print('Wild-type codon:', wt_codon)
        wt_amino_acid = translate(wt_codon)
        print('Wild-type amino acid:', wt_amino_acid)
    
    else:
        print('ISP reverse primer is closest to the target codon so target codon will be reverse complemented before translation.')
        print('Codon before reverse complement:', codon_seq)
        wt_codon = reverse_complement(codon_seq).upper()
        print('Codon after reverse complement:', wt_codon)
        print('Wild-type codon:', wt_codon)
        wt_amino_acid = translate(wt_codon)
        print('Wild-type amino acid:', wt_amino_acid)
        
    # get primer sequences for off target checks
    fwd_primer_seq = extract_slice(seq = record.seq, coord1 = fwd_feat.location.start, coord2 = fwd_feat.location.end, strand = 'Forward')
    rev_primer_seq = extract_slice(seq = record.seq, coord1 = rev_feat.location.start, coord2 = rev_feat.location.end, strand = 'Reverse')
    print('Forward primer sequence:', fwd_primer_seq)
    print('Reverse primer sequence:', rev_primer_seq)
    # Now perform logic to check which primer matches the recodonised region.
    if isp_primer_strand == 'Forward':
        isp_primer = fwd_primer_seq
        non_isp_primer = rev_primer_seq
    else:
        isp_primer = rev_primer_seq
        non_isp_primer = fwd_primer_seq 
    
    return {
        'protein_strand': protein_strand,
        'sequencing_strand': isp_primer_strand,
        'inn_prefix': inn_prefix,
        'Codon of interest': codon_seq,
        'inn_suffix': inn_suffix,
        'inn_len': len(inn_prefix) + len(codon_seq) + len(inn_suffix),
        'isp_prefix': isp_prefix,
        'Codon of interest.1': codon_seq,
        'isp_suffix': isp_suffix,
        'isp_len': len(isp_prefix) + len(codon_seq) + len(isp_suffix),
        'wt_codon': wt_codon,
        'wt_amino_acid': wt_amino_acid,
        'integration_specific_primer_for_off_target_checks': isp_primer,
        'non_integration_specific_primer_for_off_target_checks': non_isp_primer
    }, 'Success'



In [ ]:
# File input
GENBANK_DIR = "<Folder_name_here>/" #Folder name here 

variant_codon_df = pd.read_excel('variant_specific_dataframe.xlsx') #Optional dataframe with with Uniprot_ID_Amino_acid_position and Variant_codon as columns

for file in os.listdir(GENBANK_DIR):
        extracted_data = []
        name = os.path.splitext(file)[0]  # Get filename without extension
        print(f"Processing {name}...")
        uniprot_id = name.split('_')[0]  # Assuming UniProt ID is the first part of the filename  
        print(f"Fetching genomic information for UniProt ID: {uniprot_id}")
        protein_info = extract_genomic_information_from_uniprot_id(uniprot_id)
        print(f"Genomic information: {protein_info.shape[0]} records found for {uniprot_id}")
        protein_info = protein_info.loc[[0]]
        display(protein_info)
        protein_strand = protein_info['gnCoordinate.genomicLocation.reverseStrand'].values[0]
        if protein_strand:
            protein_strand = 'Reverse'
        else:
            protein_strand = 'Forward'
        print(f"Protein strand: {protein_strand}")
        data, status = process_genbank(name, GENBANK_DIR=GENBANK_DIR, read_length=150, protein_strand = protein_strand)
        if data:
            extracted_data.append(data)
            # Create df and format csv output
            extracted_df = pd.DataFrame(extracted_data)
            extracted_df.rename(columns={'Codon of interest.1': 'Codon of interest'}, inplace=True)
            
            #Add specific variant codon and translation of variant codon.
            if variant_codon_df is not variant_codon_df.empty:
                variant_codon = variant_codon_df.loc[variant_codon_df['Uniprot_ID_Amino_acid_position'] == name, 'Variant_codon'].values[0]
                print('Variant codon',variant_codon)
                extracted_df['variant_codon'] = variant_codon
                variant_amino_acid = translate(variant_codon.upper())
                print('Variant amino acid:', variant_amino_acid)
                extracted_df['variant_amino_acid'] = variant_amino_acid
            display(extracted_df)
            extracted_df.to_excel(f"Pathogenic_fastq_inputs/{name}.xlsx", index=False)
            
        else:
            print(f"Warning for {name}: {status}")

        print(f"Finished processing {name}\n")
            

        

FileNotFoundError: [Errno 2] No such file or directory: 'variant_specific_dataframe.xlsx'